In [ ]:
import pandas as pd
import numpy as np
import re
import torch
from torch.utils.data import Dataset, DataLoader
from collections import Counter
from sklearn.model_selection import train_test_split
import torch.nn as nn
import time
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

In [ ]:
# 1. Load the data and drop the 1 empty row
df = pd.read_csv('completeSpamAssassin.csv')
df = df.dropna()

In [ ]:
# 2. Custom Text Cleaning Function
def clean_text(text):
    text = str(text).lower() # Convert to lowercase
    text = re.sub(r'<[^>]+>', ' ', text) # Strip out all HTML tags
    text = re.sub(r'http\S+|www\S+|https\S+', 'http', text, flags=re.MULTILINE) # Replace all URLs with the word "http"
    text = re.sub(r'\W+', ' ', text) # Remove special characters and punctuation
    text = re.sub(r'\s+', ' ', text).strip() # Remove extra whitespace
    return text

print("Cleaning raw emails")
df['Clean_Body'] = df['Body'].apply(clean_text)

print(f"Data cleaned. Total valid emails: {len(df)}")
print("\nSample Cleaned Spam Email:")
print(df[df['Label'] == 1]['Clean_Body'].iloc[0][:150], "...")

Cleaning raw emails
Data cleaned. Total valid emails: 6045

Sample Cleaned Spam Email:
save up to 70 on life insurance why spend more than you have to life quote savings ensuring your family s financial security is very important life qu ...


In [ ]:
# 3. Build a Custom Vocabulary
print("Scanning emails to build vocabulary...")
all_words = []
for text in df['Clean_Body']:
    all_words.extend(text.split())

Scanning emails to build vocabulary...


In [ ]:
# Keep only the top 10,000 most common words to save memory
max_vocab_size = 10000
word_counts = Counter(all_words)
common_words = word_counts.most_common(max_vocab_size)

# Create a dictionary mapping each word to a number.
# We start counting at 2 to save room for special tokens.
vocab = {word: i + 2 for i, (word, count) in enumerate(common_words)}
vocab['<PAD>'] = 0  # Padding token (for short emails)
vocab['<UNK>'] = 1  # Unknown token (for rare words not in our top 10,000)

print(f"Vocabulary size: {len(vocab)} words.")

Vocabulary size: 10002 words.


In [ ]:
# 4. Tokenization and Padding Function
def text_to_sequence(text, vocab, max_len=150):
    tokens = text.split()
    # If a word is in vocab, get its ID. If not, give it the <UNK> ID (1).
    seq = [vocab.get(word, 1) for word in tokens]

    # Pad short emails with 0s, or truncate long emails to 150 words
    if len(seq) < max_len:
        seq = seq + [0] * (max_len - len(seq))
    else:
        seq = seq[:max_len]

    return seq

In [ ]:
# Apply our custom tokenizer to the dataset
max_sequence_length = 150
df['Sequence'] = df['Clean_Body'].apply(lambda x: text_to_sequence(x, vocab, max_sequence_length))

print("Custom tokenization and padding complete.")

Custom tokenization and padding complete.


In [ ]:
# 5. Build a Custom PyTorch Dataset
class SpamDataset(Dataset):
    def __init__(self, sequences, labels):
        self.sequences = torch.tensor(sequences, dtype=torch.long)
        # We use float32 for labels because our loss function (BCE) requires floats, not integers
        self.labels = torch.tensor(labels, dtype=torch.float32)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return self.sequences[idx], self.labels[idx]

In [ ]:
# Extract sequences and labels
X = df['Sequence'].tolist()
y = df['Label'].tolist()

# Split 80% for training, 20% for testing
# stratify=y ensures both sets have the exact same ratio of spam to normal emails
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Create DataLoaders
batch_size = 32
train_dataset = SpamDataset(X_train, y_train)
test_dataset = SpamDataset(X_test, y_test)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

print(f"Training batches: {len(train_loader)}")
print(f"Testing batches: {len(test_loader)}")

Training batches: 152
Testing batches: 38


In [ ]:
class SpamLSTM(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, output_dim, n_layers, drop_prob=0.2):
        super(SpamLSTM, self).__init__()

        self.hidden_dim = hidden_dim
        self.n_layers = n_layers

        # 1. Embedding Layer: Converts integer word IDs into dense mathematical vectors
        # padding_idx=0 tells the model to ignore our <PAD> zeros
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)

        # 2. The LSTM Layer: The core recurrent engine
        self.lstm = nn.LSTM(embed_dim, hidden_dim, n_layers,
                            dropout=drop_prob, batch_first=True)

        # 3. Dropout Layer: Prevents overfitting by randomly turning off neurons
        self.dropout = nn.Dropout(drop_prob)

        # 4. Fully Connected (Linear) Layer: The actual classification head
        self.fc = nn.Linear(hidden_dim, output_dim)

        # 5. Sigmoid Activation: Squashes the final number into a probability between 0 and 1
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        # x is our input batch of shape (batch_size, sequence_length)

        # Step A: Pass text through the embedding layer
        embedded = self.embedding(x)

        # Step B: Pass embeddings through the LSTM
        # lstm_out contains all hidden states. (hidden, cell) is just the final state.
        lstm_out, (hidden, cell) = self.lstm(embedded)

        # Step C: Extract the output from the VERY LAST word in the sequence
        # We only care about the network's thought process after it reads the entire email
        final_state = lstm_out[:, -1, :]

        # Step D: Pass through dropout and the final classification head
        out = self.dropout(final_state)
        out = self.fc(out)

        # Step E: Squash to a probability (0.0 to 1.0)
        out = self.sigmoid(out)

        # Return a clean 1D tensor of probabilities
        return out.squeeze()

In [ ]:
# 1. Define the Hyperparameters
vocab_size = len(vocab)
embed_dim = 100       # Each word becomes a 100-dimensional mathematical vector
hidden_dim = 128      # The size of the LSTM's "memory" cell
output_dim = 1        # Just 1 output node (probability of being Spam)
n_layers = 2          # Stacking 2 LSTMs on top of each other for deeper learning

In [ ]:
# 2. Initialize the Model and move it to the GPU
model = SpamLSTM(vocab_size, embed_dim, hidden_dim, output_dim, n_layers)
device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')
model.to(device)

# 3. Define the Loss Function
# BCELoss (Binary Cross Entropy) is the mathematical standard for binary (0 or 1) classification
criterion = nn.BCELoss()

# 4. Define the Optimizer
# We use standard Adam here. (AdamW is usually reserved for massive Transformers)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

print(f"Model initialized and moved to {device}!")
print(model) # This will print a summary of the layers you just built

Model initialized and moved to cuda!
SpamLSTM(
  (embedding): Embedding(10002, 100, padding_idx=0)
  (lstm): LSTM(100, 128, num_layers=2, batch_first=True, dropout=0.2)
  (dropout): Dropout(p=0.2, inplace=False)
  (fc): Linear(in_features=128, out_features=1, bias=True)
  (sigmoid): Sigmoid()
)


In [ ]:
epochs = 5

# Put the model into training mode
model.train()

print("Starting the Training Loop...")

for epoch in range(epochs):
    start_time = time.time()
    total_train_loss = 0

    # Iterate through our DataLoader batch by batch
    for step, (batch_seqs, batch_labels) in enumerate(train_loader):

        # 1. Move tensors to GPU
        batch_seqs = batch_seqs.to(device)
        batch_labels = batch_labels.to(device)

        # 2. Clear old gradients
        model.zero_grad()

        # 3. Forward Pass: Feed the data to the model
        predictions = model(batch_seqs)

        # 4. Calculate Loss
        # We compare the model's single probability output to the actual 0/1 label
        loss = criterion(predictions, batch_labels)
        total_train_loss += loss.item()

        # 5. Backward Pass (Backpropagation)
        loss.backward()

        # 6. Update Weights
        optimizer.step()

        # Print a mini-update every 50 batches
        if step % 50 == 0 and step > 0:
            print(f"  Batch {step:>3} / {len(train_loader)}.  Loss: {loss.item():.4f}")

    # Calculate average loss for the epoch
    avg_train_loss = total_train_loss / len(train_loader)
    elapsed = time.time() - start_time

    print(f"\n======== Epoch {epoch + 1} / {epochs} Completed ========")
    print(f"  Average Training Loss: {avg_train_loss:.4f}")
    print(f"  Epoch Time: {elapsed:.2f}s\n")

Starting the Training Loop...
  Batch  50 / 152.  Loss: 0.5441
  Batch 100 / 152.  Loss: 0.6103
  Batch 150 / 152.  Loss: 0.4972

======== Epoch 1 / 5 Completed ========
  Average Training Loss: 0.5379
  Epoch Time: 2.17s

  Batch  50 / 152.  Loss: 0.5085
  Batch 100 / 152.  Loss: 0.6669
  Batch 150 / 152.  Loss: 0.6568

======== Epoch 2 / 5 Completed ========
  Average Training Loss: 0.5854
  Epoch Time: 0.74s

  Batch  50 / 152.  Loss: 0.5864
  Batch 100 / 152.  Loss: 0.6366
  Batch 150 / 152.  Loss: 0.5943

======== Epoch 3 / 5 Completed ========
  Average Training Loss: 0.6187
  Epoch Time: 0.75s

  Batch  50 / 152.  Loss: 0.4861
  Batch 100 / 152.  Loss: 0.4227
  Batch 150 / 152.  Loss: 0.2922

======== Epoch 4 / 5 Completed ========
  Average Training Loss: 0.5266
  Epoch Time: 0.74s

  Batch  50 / 152.  Loss: 0.2561
  Batch 100 / 152.  Loss: 0.2102
  Batch 150 / 152.  Loss: 0.2058

======== Epoch 5 / 5 Completed ========
  Average Training Loss: 0.3608
  Epoch Time: 0.74s



In [ ]:
# 1. Put the model in evaluation mode
# This turns off Dropout layers so the network uses its full brain power
model.eval()

predictions = []
true_labels = []

print("Running Evaluation on the unseen Test Set...")
start_time = time.time()

# 2. Disable gradient calculation to save memory and speed up inference
with torch.no_grad():
    for batch_seqs, batch_labels in test_loader:

        # Move tensors to GPU
        batch_seqs = batch_seqs.to(device)
        batch_labels = batch_labels.to(device)

        # Forward pass to get probabilities
        probs = model(batch_seqs)

        # 3. Apply Threshold: Convert probabilities to binary predictions (0 or 1)
        batch_preds = (probs >= 0.5).int().cpu().numpy()

        # Store predictions and true labels for scikit-learn
        predictions.extend(batch_preds)
        true_labels.extend(batch_labels.int().cpu().numpy())

# 4. Calculate Final Metrics
accuracy = accuracy_score(true_labels, predictions)
conf_matrix = confusion_matrix(true_labels, predictions)

print(f"\nEvaluation complete in {round(time.time() - start_time, 2)}s!\n")
print(f"=====================================")
print(f" Test Accuracy: {accuracy * 100:.2f}%")
print(f"=====================================\n")

print("Confusion Matrix:")
print(conf_matrix)

print("\nClassification Report:")
print(classification_report(true_labels, predictions, target_names=['Ham (0)', 'Spam (1)']))

Running Evaluation on the unseen Test Set...

Evaluation complete in 0.18s!

 Test Accuracy: 90.98%

Confusion Matrix:
[[760  70]
 [ 39 340]]

Classification Report:
              precision    recall  f1-score   support

     Ham (0)       0.95      0.92      0.93       830
    Spam (1)       0.83      0.90      0.86       379

    accuracy                           0.91      1209
   macro avg       0.89      0.91      0.90      1209
weighted avg       0.91      0.91      0.91      1209



In [ ]:
def predict_spam(email_text, model, vocab, max_len=150):
    # 1. Put the model in evaluation mode
    model.eval()

    # 2. Clean the raw text using the exact same function from Step 1
    cleaned = clean_text(email_text)

    # 3. Convert the cleaned text to a sequence of integer IDs and pad it
    seq = text_to_sequence(cleaned, vocab, max_len)

    # 4. Convert to PyTorch tensor, add a batch dimension of 1, and move to GPU
    # .unsqueeze(0) changes the shape from (150) to (1, 150) so it matches what the LSTM expects
    tensor_seq = torch.tensor(seq, dtype=torch.long).unsqueeze(0).to(device)

    # 5. Run it through the network without calculating gradients
    with torch.no_grad():
        # .item() pulls the single float value out of the PyTorch tensor
        prob = model(tensor_seq).item()

    # 6. Apply our threshold and translate to English
    if prob >= 0.5:
        return f"SPAM (Confidence: {prob * 100:.2f}%)"
    else:
        return f"HAM / Normal (Confidence: {(1 - prob) * 100:.2f}%)"

In [ ]:
test_email_1 = "Congratulations! You have been selected to win a free $1,000 Walmart gift card. Click here to claim your prize now!"
test_email_2 = "Hey, are we still on for the code review tomorrow at 10 AM? Let me know if you need to reschedule."
test_email_3 = "URGENT: Your account has been compromised. Please click this link and provide your password immediately to restore access."
test_email_4 = "Can you send me the latest version of the dataset? I need it for the pipeline test."

print(f"Email 1:\n{predict_spam(test_email_1, model, vocab)}\n")
print(f"Email 2:\n{predict_spam(test_email_2, model, vocab)}\n")
print(f"Email 3:\n{predict_spam(test_email_3, model, vocab)}\n")
print(f"Email 4:\n{predict_spam(test_email_4, model, vocab)}\n")

Email 1:
SPAM (Confidence: 92.29%)

Email 2:
HAM / Normal (Confidence: 96.85%)

Email 3:
SPAM (Confidence: 92.29%)

Email 4:
SPAM (Confidence: 68.84%)

